# 08. 코드 공유 1위 코드 재현 + 디벨롭

**원본**: 데이콘 코드 공유 「score : 493643.24145 / 그냥 한번 끄적여봅니다....」 (2021, Public 493,643)

| 원본이 한 것 | 내용 |
| --- | --- |
| 배급사 정리 | `(주)` · 특수문자 제거 후 CJ/CGV → CJ 등 같은 회사 묶기 |
| 랭크 인코딩 | 장르 = 관객수 **평균** 순위, 배급사 = 관객수 **중앙값** 순위 (`num_rank`, 처음 보는 배급사는 0) |
| 변수 6개 | `num_rank` · `time` · `num_staff` · `log1p(num_actor)` · `genre_rank` · 관람 등급 더미 |
| 타깃 | `log1p(관객수)` |
| 모델 6개 평균 | GradientBoosting · NGBoost · LightGBM · XGBoost · CatBoost · RandomForest, 10-Fold 각 fold 모델의 test 예측 평균 |

**원본의 문제점**
1. CV 점수 계산 버그: 정답은 log 단위, 예측은 원래 단위로 비교 → 출력된 CV 점수가 의미 없음
2. 랭크 인코딩을 train 전체 정답으로 만든 뒤 CV → CV가 실제보다 좋게 나옴 (test 예측에는 문제 없음)

| 단계 | 내용 | 제출 파일 |
| --- | --- | --- |
| 1 | **원본 그대로 재현** (CV만 올바르게 다시 계산) | `08_1_reproduce.csv` |
| 2 | **디벨롭**: 랭크 인코딩을 fold 안에서 · 우리 EDA 변수 추가 · 타깃 log vs sqrt | |
| 3 | 6개 모델 + TabPFN 가중 평균 (가중치는 OOF로 결정) | `08_3_final.csv` |

- 디벨롭 단계 검증: 팀 공통 5-Fold (seed 42), 원래 관객수 단위 RMSE → 01~07과 비교 가능

## 0. 설정

In [1]:
import os
import re
import time
import pickle
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import nnls
from sklearn import set_config
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from ngboost import NGBRegressor

from features import make_features

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
y = train['box_off_num']
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5

## 1. 원본 전처리 (그대로)

In [2]:
def clean_distributor(s):
    s = s.str.replace('(주)', '', regex=False)
    return s.map(lambda x: re.sub(r'[^0-9a-zA-Z가-힣]', '', x))


def get_dis(x):
    if 'CJ' in x or 'CGV' in x: return 'CJ'
    elif '쇼박스' in x: return '쇼박스'
    elif 'SK' in x: return 'SK'
    elif '리틀빅픽' in x: return '리틀빅픽처스'
    elif '스폰지' in x: return '스폰지'
    elif '싸이더스' in x: return '싸이더스'
    elif '에이원' in x: return '에이원'
    elif '마인스' in x: return '마인스'
    elif '마운틴픽' in x: return '마운틴픽처스'
    elif '디씨드' in x: return '디씨드'
    elif '드림팩트' in x: return '드림팩트'
    elif '메가박스' in x: return '메가박스'
    elif '마운틴' in x: return '마운틴'
    else: return x


for df in (train, test):
    df['dist_clean'] = clean_distributor(df['distributor']).map(get_dis)


def rank_encode(tr_df, tr_y, ap_df):
    """원본 랭크 인코딩: tr_df의 정답으로 순위를 만들어 ap_df에 적용"""
    tmp = tr_df.assign(_y=tr_y.values)
    g_rank = tmp.groupby('genre')['_y'].mean().rank(method='first').astype(int)
    d_rank = tmp.groupby('dist_clean')['_y'].median().rank(method='first').astype(int)
    return ap_df['genre'].map(g_rank).fillna(0).values, ap_df['dist_clean'].map(d_rank).fillna(0).values


def original_X(tr_df, tr_y, ap_df):
    g, d = rank_encode(tr_df, tr_y, ap_df)
    X = pd.DataFrame({'num_rank': d, 'time': ap_df['time'].values, 'num_staff': ap_df['num_staff'].values,
                      'num_actor': np.log1p(ap_df['num_actor'].values), 'genre_rank': g,
                      'screening_rat': ap_df['screening_rat'].values})
    X = pd.get_dummies(X, columns=['screening_rat'], dtype=float)
    for r in ['12세 관람가', '15세 관람가', '전체 관람가', '청소년 관람불가']:
        if f'screening_rat_{r}' not in X:
            X[f'screening_rat_{r}'] = 0.0
    return X[sorted(X.columns)].astype(float)


print('정리 후 배급사 수:', train['dist_clean'].nunique())
display(original_X(train, y, train).head())

정리 후 배급사 수: 147


,genre_rank,num_actor,num_rank,num_staff,screening_rat_12세 관람가,screening_rat_15세 관람가,screening_rat_전체 관람가,screening_rat_청소년 관람불가,time
0,11.0,1.098612,134.0,91.0,0.0,0.0,0.0,1.0,96.0
1,12.0,1.386294,145.0,387.0,0.0,0.0,0.0,1.0,130.0
2,11.0,1.609438,145.0,343.0,0.0,1.0,0.0,0.0,123.0
3,9.0,1.945910,140.0,20.0,0.0,0.0,1.0,0.0,101.0
4,9.0,1.098612,145.0,251.0,0.0,1.0,0.0,0.0,108.0


## 2. 모델 6개 (원본 설정 그대로)

In [3]:
MODELS = {
    'GBM':  lambda: GradientBoostingRegressor(random_state=42),
    'NGB':  lambda: NGBRegressor(random_state=518, verbose=False),
    'LGBM': lambda: LGBMRegressor(random_state=518, verbose=-1),
    'XGB':  lambda: XGBRegressor(random_state=518),
    'CAT':  lambda: CatBoostRegressor(random_state=518, silent=True),
    'RF':   lambda: RandomForestRegressor(random_state=518),
}
TARGETS = {
    'log':  (np.log1p, lambda p: np.expm1(np.clip(p, 0, None))),   # 원본: 음수 예측은 0으로
    'sqrt': (np.sqrt,  lambda p: np.square(np.clip(p, 0, None))),
}

## 3. 1단계: 원본 그대로 재현
원본과 똑같이 10-Fold, 랭크 인코딩은 train 전체로, 각 fold 모델의 test 예측을 평균.
CV는 **원래 관객수 단위로 올바르게** 다시 계산 (원본 출력 CV는 버그라 비교 불가).

In [4]:
kf10 = KFold(n_splits=10, shuffle=True, random_state=42)
X_full = original_X(train, y, train)      # 원본처럼 train 전체 정답으로 랭크 (CV가 낙관적)
X_te = original_X(train, y, test)
fwd, inv = TARGETS['log']

repro_test = {}
repro_cv = {}
t0 = time.time()
for name, mk in MODELS.items():
    oof = np.zeros(len(y)); te_pred = np.zeros(len(test))
    for tr_idx, va_idx in kf10.split(X_full):
        m = mk(); m.fit(X_full.iloc[tr_idx], fwd(y.iloc[tr_idx]))
        oof[va_idx] = inv(m.predict(X_full.iloc[va_idx]))
        te_pred += inv(m.predict(X_te)) / 10
    repro_test[name] = te_pred
    repro_cv[name] = rmse(y, oof)
    print(f'{name:<5} CV RMSE (원래 단위) {repro_cv[name]:>11,.0f}')

repro_pred = np.mean(list(repro_test.values()), axis=0)
sub = submission.copy(); sub['box_off_num'] = repro_pred
sub.to_csv('08_1_reproduce.csv', index=False)
print(f'\n저장: 08_1_reproduce.csv | 예측 평균 {repro_pred.mean():,.0f} · 최대 {repro_pred.max():,.0f} | {time.time()-t0:.0f}초')
print('→ 원본 Public 493,643은 2021년 리더보드 기준. 지금 리더보드에서 이 파일 점수를 확인해서 재현 여부 판단')

GBM   CV RMSE (원래 단위)   1,617,836


NGB   CV RMSE (원래 단위)   1,476,973


LGBM  CV RMSE (원래 단위)   1,546,681


XGB   CV RMSE (원래 단위)   1,653,784


CAT   CV RMSE (원래 단위)   1,480,457


RF    CV RMSE (원래 단위)   1,534,794

저장: 08_1_reproduce.csv | 예측 평균 486,665 · 최대 6,694,309 | 53초
→ 원본 Public 493,643은 2021년 리더보드 기준. 지금 리더보드에서 이 파일 점수를 확인해서 재현 여부 판단


## 4. 2단계: 디벨롭
팀 공통 5-Fold (seed 42)로 바꾸고, 아래를 하나씩 비교.

| 실험 | 내용 |
| --- | --- |
| D1 | 원본 변수 · **랭크 인코딩을 fold 안에서** (정직한 CV) · log |
| D2 | D1 + **우리 EDA 변수** (데뷔작 · 감독 이전 기록 · 개봉 시기 · 관람 등급 · 배급사 편수/4대) · log |
| D3 | D2 · **sqrt 타깃** |

우리 EDA 변수는 04~06에서 TabPFN 점수를 올린 것들. GBM은 결측을 못 받아서 `dir_prev_bfnum`은 0 + 데뷔작 플래그로 처리.

In [5]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
CACHE_DIR = 'cache_08'
os.makedirs(CACHE_DIR, exist_ok=True)

# 우리 EDA 변수 (features.py, 04~06에서 효과 있던 그룹)
EXTRA_OPTS = {'debut': 1, 'debut_mode': 'zero_flag', 'distributor': 1, 'date': 1, 'rating': 1}
EXTRA_COLS = ['dir_prev_bfnum', 'is_debut', 'dir_prev_num', '년', '월', 'weekday', 'quarter', 'is_peak',
              'is_15', 'is_all', 'staff_x_rat0', 'staff_x_rat1', 'staff_x_rat2', 'staff_x_rat3', 'dist_count', 'is_big4']
EX_tr, EX_te = make_features(train, test, EXTRA_OPTS)
EX_tr, EX_te = EX_tr[EXTRA_COLS].reset_index(drop=True), EX_te[EXTRA_COLS].reset_index(drop=True)


def build(tr_idx, ap_df, ap_extra, use_extra):
    X = original_X(train.iloc[tr_idx], y.iloc[tr_idx], ap_df).reset_index(drop=True)
    return pd.concat([X, ap_extra], axis=1) if use_extra else X


def run(name, use_extra, target):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    fwd, inv = TARGETS[target]
    oofs = {m: np.zeros(len(y)) for m in MODELS}
    tests = {m: np.zeros(len(test)) for m in MODELS}
    t0 = time.time()
    for tr_idx, va_idx in kf5.split(train):
        Xtr = build(tr_idx, train.iloc[tr_idx], EX_tr.iloc[tr_idx].reset_index(drop=True), use_extra)
        Xva = build(tr_idx, train.iloc[va_idx], EX_tr.iloc[va_idx].reset_index(drop=True), use_extra)
        Xte = build(tr_idx, test, EX_te, use_extra)
        for mname, mk in MODELS.items():
            m = mk(); m.fit(Xtr, fwd(y.iloc[tr_idx]))
            oofs[mname][va_idx] = inv(m.predict(Xva))
            tests[mname] += inv(m.predict(Xte)) / 5
    avg = np.mean(list(oofs.values()), axis=0)
    res = {'실험': name, '타깃': target, '변수 수': Xtr.shape[1],
           **{f'{m}': rmse(y, o) for m, o in oofs.items()},
           '6개 평균': rmse(y, avg), '6개 평균 흥행작': rmse(y[hit], avg[hit]),
           '소요(초)': time.time() - t0, 'oofs': oofs, 'tests': tests}
    pickle.dump(res, open(path, 'wb'))
    return res


D = {}
for name, use_extra, target in [('D1_orig_log', False, 'log'), ('D2_extra_log', True, 'log'), ('D3_extra_sqrt', True, 'sqrt')]:
    D[name] = r = run(name, use_extra, target)
    print(f"{name:<14} 변수 {r['변수 수']:>2} | 6개 평균 CV RMSE {r['6개 평균']:>11,.0f} | 흥행작 {r['6개 평균 흥행작']:>11,.0f} | {r['소요(초)']:.0f}초")

D1_orig_log    변수  9 | 6개 평균 CV RMSE   1,497,137 | 흥행작   3,386,923 | 25초


D2_extra_log   변수 25 | 6개 평균 CV RMSE   1,441,918 | 흥행작   3,287,889 | 100초


D3_extra_sqrt  변수 25 | 6개 평균 CV RMSE   1,452,779 | 흥행작   3,193,924 | 98초


In [6]:
cols = ['실험', '타깃', '변수 수'] + list(MODELS) + ['6개 평균', '6개 평균 흥행작']
tbl = pd.DataFrame([{k: r[k] for k in cols} for r in D.values()])
display(tbl.style.format({c: '{:,.0f}' for c in list(MODELS) + ['6개 평균', '6개 평균 흥행작']})
        .highlight_min(subset=list(MODELS) + ['6개 평균'], color='#c6efce'))
best_D = min(D.values(), key=lambda r: r['6개 평균'])
print('→ 디벨롭 최고:', best_D['실험'], f"(6개 평균 CV {best_D['6개 평균']:,.0f})")
print('참고: TabPFN v3.5 (06 최종) CV 1,350,295')

,실험,타깃,변수 수,GBM,NGB,LGBM,XGB,CAT,RF,6개 평균,6개 평균 흥행작
0,D1_orig_log,log,9,"1,530,920","1,497,454","1,511,077","1,670,510","1,459,108","1,575,720","1,497,137","3,386,923"
1,D2_extra_log,log,25,"1,496,049","1,485,034","1,440,448","1,498,884","1,454,155","1,546,086","1,441,918","3,287,889"
2,D3_extra_sqrt,sqrt,25,"1,494,176","1,490,051","1,383,923","1,613,724","1,433,549","1,468,297","1,452,779","3,193,924"


→ 디벨롭 최고: D2_extra_log (6개 평균 CV 1,441,918)
참고: TabPFN v3.5 (06 최종) CV 1,350,295


## 5. 3단계: 6개 모델 + TabPFN 가중 평균
- 원본처럼 단순 평균 대신, **OOF 예측으로 가중치를 정함** (음수 없는 최소제곱 = NNLS, 합이 1이 되게 정규화)
- TabPFN은 06 최종 결과(같은 5-Fold OOF)와 `06_M_nan.csv`의 test 예측을 사용
- 가중치를 같은 OOF로 고르기 때문에 CV가 약간 낙관적일 수 있음 → 단순 평균과 같이 비교

In [7]:
members_oof = dict(best_D['oofs'])
members_test = dict(best_D['tests'])
if os.path.exists('cache_06/M_nan.pkl') and os.path.exists('06_M_nan.csv'):
    members_oof['TabPFN'] = pickle.load(open('cache_06/M_nan.pkl', 'rb'))['oof']
    members_test['TabPFN'] = pd.read_csv('06_M_nan.csv')['box_off_num'].values

names = list(members_oof)
O = np.column_stack([members_oof[n] for n in names])
T = np.column_stack([members_test[n] for n in names])

w, _ = nnls(O, y.values.astype(float))
w = w / w.sum()
blend_oof, blend_test = O @ w, T @ w
simple_oof, simple_test = O.mean(axis=1), T.mean(axis=1)

display(pd.DataFrame({'모델': names, '단독 CV RMSE': [rmse(y, members_oof[n]) for n in names], 'NNLS 가중치': w})
        .sort_values('NNLS 가중치', ascending=False)
        .style.format({'단독 CV RMSE': '{:,.0f}', 'NNLS 가중치': '{:.3f}'}))

summary = pd.DataFrame([
    {'방법': '원본 방식 (6개 단순 평균, D 최고)', 'CV RMSE': best_D['6개 평균']},
    {'방법': f'{len(names)}개 단순 평균', 'CV RMSE': rmse(y, simple_oof)},
    {'방법': f'{len(names)}개 NNLS 가중 평균', 'CV RMSE': rmse(y, blend_oof)},
    {'방법': 'TabPFN 단독 (06)', 'CV RMSE': rmse(y, members_oof['TabPFN']) if 'TabPFN' in members_oof else np.nan},
])
summary['흥행작 RMSE'] = [rmse(y[hit], o[hit]) for o in [np.mean(list(best_D['oofs'].values()), axis=0), simple_oof, blend_oof,
                                                       members_oof.get('TabPFN', simple_oof)]]
display(summary.style.format({'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))

,모델,단독 CV RMSE,NNLS 가중치
6,TabPFN,"1,350,295",0.651
4,CAT,"1,454,155",0.211
2,LGBM,"1,440,448",0.138
1,NGB,"1,485,034",0.000
0,GBM,"1,496,049",0.000
3,XGB,"1,498,884",0.000
5,RF,"1,546,086",0.000


,방법,CV RMSE,흥행작 RMSE
0,"원본 방식 (6개 단순 평균, D 최고)","1,441,918","3,287,889"
1,7개 단순 평균,"1,418,143","3,229,497"
2,7개 NNLS 가중 평균,"1,351,383","3,055,060"
3,TabPFN 단독 (06),"1,350,295","3,025,307"


## 6. 최종 제출 파일

In [8]:
final = np.clip(blend_test, 0, None)
sub = submission.copy(); sub['box_off_num'] = final
sub.to_csv('08_3_final.csv', index=False)
print(f'저장: 08_3_final.csv (NNLS 가중 평균) | 예측 평균 {final.mean():,.0f} · 최대 {final.max():,.0f}')

sub2 = submission.copy(); sub2['box_off_num'] = np.clip(simple_test, 0, None)
sub2.to_csv('08_3_simple_avg.csv', index=False)
print('저장: 08_3_simple_avg.csv (단순 평균, 비교용)')

pd.concat([tbl, summary.rename(columns={'방법': '실험'})], ignore_index=True).to_csv('08_results.csv', index=False, encoding='utf-8-sig')
print('08_results.csv 저장')

저장: 08_3_final.csv (NNLS 가중 평균) | 예측 평균 594,339 · 최대 9,957,634
저장: 08_3_simple_avg.csv (단순 평균, 비교용)
08_results.csv 저장
